## 0. Setup

In [1]:
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support, confusion_matrix

!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

SEED = 42

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Bagian 1. Data Audit
Split 80:20 stratified (`random_state=42`) dari `train_preprocess.tsv`; `valid_preprocess.tsv` hanya untuk final evaluation.

In [2]:
print("DATA AUDIT")
base_url = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/"
train_url = base_url + "train_preprocess.tsv"
valid_url = base_url + "valid_preprocess.tsv"

if not os.path.exists("train_preprocess.tsv"):
    urllib.request.urlretrieve(train_url, "train_preprocess.tsv")
if not os.path.exists("valid_preprocess.tsv"):
    urllib.request.urlretrieve(valid_url, "valid_preprocess.tsv")

df_raw_train = pd.read_csv("train_preprocess.tsv", sep="\t", header=None, names=["text", "label"])
df_final_eval = pd.read_csv("valid_preprocess.tsv", sep="\t", header=None, names=["text", "label"])

df_train, df_dev = train_test_split(
    df_raw_train, test_size=0.20, random_state=SEED, stratify=df_raw_train["label"]
)

print(f"1. Jumlah Data:")
print(f"   - Raw Train Set : {len(df_raw_train)}")
print(f"   - Split Train   : {len(df_train)}")
print(f"   - Split Dev     : {len(df_dev)}")
print(f"   - Final Eval    : {len(df_final_eval)}\n")

print("2. Distribusi Kelas (Train Set):")
print(df_train['label'].value_counts())
print("\n3. Missing Value (Train Set):", df_train.isnull().sum().to_dict())
print("4. Duplicate (Train Set):", df_train.duplicated(subset=['text']).sum())

df_train['word_len'] = df_train['text'].apply(lambda x: len(x.split()))
print("\n5. Distribusi Panjang Teks (Jumlah Kata):")
print(df_train['word_len'].describe())

print("\n6. Contoh 5 Teks per Kelas:")
for label in ['positive', 'neutral', 'negative']:
    print(f"\n--- KELAS: {label.upper()} ---")
    samples = df_train[df_train['label'] == label]['text'].head(5)
    for i, t in enumerate(samples, 1):
        print(f" [{i}] {t}")

print("\nDistribusi kelas per subset:")
for nm, d in [("Raw train", df_raw_train), ("Train 80%", df_train), ("Dev 20%", df_dev), ("Final eval", df_final_eval)]:
    print(f"  {nm:<11}: {d['label'].value_counts().to_dict()}")

DATA AUDIT
1. Jumlah Data:
   - Raw Train Set : 11000
   - Split Train   : 8800
   - Split Dev     : 2200
   - Final Eval    : 1260

2. Distribusi Kelas (Train Set):
label
positive    5133
negative    2749
neutral      918
Name: count, dtype: int64

3. Missing Value (Train Set): {'text': 0, 'label': 0}
4. Duplicate (Train Set): 42

5. Distribusi Panjang Teks (Jumlah Kata):
count    8800.000000
mean       33.003750
std        21.300807
min         1.000000
25%        15.000000
50%        28.000000
75%        48.000000
max       110.000000
Name: word_len, dtype: float64

6. Contoh 5 Teks per Kelas:

--- KELAS: POSITIVE ---
 [1] tidak menyesal deh pakai bagus skin care hasil terpampang nyata tanpa bedak tanpa menguras :
 [2] makanan nya mostly adalah makanan barat , harga nya memang cukup menguras kantong . tapi memang restoran ini sangat romantis dengan pemandangan nya yang sangat indah di malam hari
 [3] kalau anda ingin makanan ringan sebagai camilan . dapur pasta menyediakan keperluan

## Fungsi evaluasi
`labels=` dibuat eksplisit agar urutan positive, neutral, negative tidak tertukar pada classification report.

In [3]:
results_list = []
LABELS = ["positive", "neutral", "negative"]

def evaluate_pipeline(y_true, y_pred, exp_id, description, stage="Tahap A"):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=LABELS, zero_division=0)
    macro_f1 = f1.mean()
    res = {"Stage": stage, "Exp_ID": exp_id, "Description": description,
           "Accuracy": round(acc, 4), "Macro_F1": round(macro_f1, 4),
           "Prec_Pos": round(prec[0], 4), "Rec_Pos": round(rec[0], 4), "F1_Pos": round(f1[0], 4),
           "Prec_Neu": round(prec[1], 4), "Rec_Neu": round(rec[1], 4), "F1_Neu": round(f1[1], 4),
           "Prec_Neg": round(prec[2], 4), "Rec_Neg": round(rec[2], 4), "F1_Neg": round(f1[2], 4)}
    results_list.append(res)
    print(f"\n=== [{exp_id}] {description} ({stage}) ===")
    print(f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
    print(classification_report(y_true, y_pred, labels=LABELS, target_names=LABELS, digits=4, zero_division=0))
    print("Confusion Matrix (baris=gold, kolom=prediksi; urutan positive, neutral, negative):")
    print(confusion_matrix(y_true, y_pred, labels=LABELS))
    return res

## Tahap A. Ablation Study (classifier tetap: Logistic Regression)
EXP-0 baseline, EXP-1 normalisasi, EXP-2 stopword (negasi dipertahankan), EXP-3 stemming, EXP-4 n-gram.

In [4]:
print("TAHAP A (ABLATION STUDY)")


def prep_minimal(text):
    return re.sub(r'\s+', ' ', text.lower()).strip()

# EXP-0
X_tr_0 = df_train['text'].apply(prep_minimal)
X_dev_0 = df_dev['text'].apply(prep_minimal)
vec_0 = TfidfVectorizer(ngram_range=(1,1))
clf_0 = LogisticRegression(max_iter=1000, random_state=SEED).fit(vec_0.fit_transform(X_tr_0), df_train['label'])
evaluate_pipeline(df_dev['label'], clf_0.predict(vec_0.transform(X_dev_0)), "EXP-0", "Baseline Minimal + Unigram + LogReg")

# EXP-1
slang_dict = {"gk": "tidak", "tdk": "tidak", "bgt": "banget", "yg": "yang", "dgn": "dengan", "gak": "tidak", "ga": "tidak"}
def prep_norm(text):
    text = prep_minimal(text)
    return " ".join([slang_dict.get(w, w) for w in text.split()])

X_tr_1 = df_train['text'].apply(prep_norm)
X_dev_1 = df_dev['text'].apply(prep_norm)
vec_1 = TfidfVectorizer(ngram_range=(1,1))
clf_1 = LogisticRegression(max_iter=1000, random_state=SEED).fit(vec_1.fit_transform(X_tr_1), df_train['label'])
evaluate_pipeline(df_dev['label'], clf_1.predict(vec_1.transform(X_dev_1)), "EXP-1", "EXP-0 + Normalization")

# EXP-2
ind_stopwords = set(stopwords.words('indonesian')) - {"tidak", "bukan", "belum", "tanpa", "jangan"}
def prep_stopword(text):
    return " ".join([w for w in prep_norm(text).split() if w not in ind_stopwords])

X_tr_2 = df_train['text'].apply(prep_stopword)
X_dev_2 = df_dev['text'].apply(prep_stopword)
vec_2 = TfidfVectorizer(ngram_range=(1,1))
clf_2 = LogisticRegression(max_iter=1000, random_state=SEED).fit(vec_2.fit_transform(X_tr_2), df_train['label'])
evaluate_pipeline(df_dev['label'], clf_2.predict(vec_2.transform(X_dev_2)), "EXP-2", "EXP-1 + Stopword")

# EXP-3
stemmer = StemmerFactory().create_stemmer()
def prep_stem(text):
    return stemmer.stem(prep_norm(text))

X_tr_3 = df_train['text'].apply(prep_stem)
X_dev_3 = df_dev['text'].apply(prep_stem)
vec_3 = TfidfVectorizer(ngram_range=(1,1))
clf_3 = LogisticRegression(max_iter=1000, random_state=SEED).fit(vec_3.fit_transform(X_tr_3), df_train['label'])
evaluate_pipeline(df_dev['label'], clf_3.predict(vec_3.transform(X_dev_3)), "EXP-3", "EXP-1 + Stemming")

# EXP-4
vec_4 = TfidfVectorizer(ngram_range=(1,2))
X_tr_4 = vec_4.fit_transform(X_tr_1)
X_dev_4 = vec_4.transform(X_dev_1)
clf_4 = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_tr_4, df_train['label'])
evaluate_pipeline(df_dev['label'], clf_4.predict(X_dev_4), "EXP-4", "Best Preprocess + TF-IDF (1,2) N-gram")

TAHAP A (ABLATION STUDY)

=== [EXP-0] Baseline Minimal + Unigram + LogReg (Tahap A) ===
Accuracy: 0.8745 | Macro-F1: 0.8428
              precision    recall  f1-score   support

    positive     0.9030    0.9291    0.9159      1283
     neutral     0.9244    0.6913    0.7910       230
    negative     0.8093    0.8341    0.8215       687

    accuracy                         0.8745      2200
   macro avg     0.8789    0.8181    0.8428      2200
weighted avg     0.8760    0.8745    0.8734      2200

Confusion Matrix (baris=gold, kolom=prediksi; urutan positive, neutral, negative):
[[1192    7   84]
 [  20  159   51]
 [ 108    6  573]]

=== [EXP-1] EXP-0 + Normalization (Tahap A) ===
Accuracy: 0.8755 | Macro-F1: 0.8439
              precision    recall  f1-score   support

    positive     0.9032    0.9306    0.9167      1283
     neutral     0.9298    0.6913    0.7930       230
    negative     0.8105    0.8341    0.8221       687

    accuracy                         0.8755      2200


{'Stage': 'Tahap A',
 'Exp_ID': 'EXP-4',
 'Description': 'Best Preprocess + TF-IDF (1,2) N-gram',
 'Accuracy': 0.8727,
 'Macro_F1': np.float64(0.8158),
 'Prec_Pos': np.float64(0.8973),
 'Rec_Pos': np.float64(0.9462),
 'F1_Pos': np.float64(0.9211),
 'Prec_Neu': np.float64(0.9692),
 'Rec_Neu': np.float64(0.5478),
 'F1_Neu': np.float64(0.7),
 'Prec_Neg': np.float64(0.8089),
 'Rec_Neg': np.float64(0.8443),
 'F1_Neg': np.float64(0.8262)}

### EXP-2b. Uji khusus negasi dibuang
Soal meminta pengujian apakah negasi sebaiknya dibuang.

In [5]:

full_sw = set(stopwords.words('indonesian'))
prep_stop_all = lambda t: " ".join(w for w in prep_norm(t).split() if w not in full_sw)
Xa, Xb = df_train['text'].apply(prep_stop_all), df_dev['text'].apply(prep_stop_all)
v2b = TfidfVectorizer()
c2b = LogisticRegression(max_iter=1000, random_state=SEED).fit(v2b.fit_transform(Xa), df_train['label'])
evaluate_pipeline(df_dev['label'], c2b.predict(v2b.transform(Xb)), "EXP-2b", "EXP-1 + Stopword (negasi ikut dibuang)")
print("Kata negasi yang ada di stopword NLTK:", sorted({"tidak","bukan","belum","tanpa","jangan"} & full_sw))


=== [EXP-2b] EXP-1 + Stopword (negasi ikut dibuang) (Tahap A) ===
Accuracy: 0.8532 | Macro-F1: 0.8001
              precision    recall  f1-score   support

    positive     0.8890    0.9299    0.9090      1283
     neutral     0.9041    0.5739    0.7021       230
    negative     0.7753    0.8035    0.7891       687

    accuracy                         0.8532      2200
   macro avg     0.8561    0.7691    0.8001      2200
weighted avg     0.8551    0.8532    0.8499      2200

Confusion Matrix (baris=gold, kolom=prediksi; urutan positive, neutral, negative):
[[1193    6   84]
 [  22  132   76]
 [ 127    8  552]]
Kata negasi yang ada di stopword NLTK: ['belum', 'bukan', 'jangan', 'tanpa', 'tidak']


## Tahap B. Model Comparison
Fitur dikunci pada **EXP-1** (pipeline terbaik Tahap A berdasarkan Macro-F1 development set). Eksperimen tambahan dengan fitur EXP-4 dilaporkan terpisah.

In [6]:
# Ganti seluruh bagian "TAHAP B" di sel lama dengan ini.
print("TAHAP B (MODEL COMPARISON) - fitur terkunci: EXP-1 (normalisasi + TF-IDF unigram)")
Xtr_B  = vec_1.transform(X_tr_1)
Xdev_B = vec_1.transform(X_dev_1)
models = {
    "M1-NB":  MultinomialNB(alpha=1.0),
    "M2-LR":  LogisticRegression(max_iter=1000, random_state=SEED),
    "M3-SVM": LinearSVC(C=1.0, random_state=SEED),
}
dev_pred = {}
for name, mdl in models.items():
    mdl.fit(Xtr_B, df_train["label"])
    dev_pred[name] = mdl.predict(Xdev_B)
    evaluate_pipeline(df_dev["label"], dev_pred[name], name, name, stage="Tahap B")

# Eksperimen tambahan: classifier yang sama dengan fitur EXP-4 (unigram+bigram)
Xtr_4b, Xdev_4b = vec_4.transform(X_tr_1), vec_4.transform(X_dev_1)
for name, mdl in {"M1-NB":MultinomialNB(alpha=1.0),
                  "M2-LR":LogisticRegression(max_iter=1000, random_state=SEED),
                  "M3-SVM":LinearSVC(C=1.0, random_state=SEED)}.items():
    mdl.fit(Xtr_4b, df_train["label"])
    evaluate_pipeline(df_dev["label"], mdl.predict(Xdev_4b), name+"-bigram", name+" fitur EXP-4",
                      stage="Tahap B (tambahan)")

TAHAP B (MODEL COMPARISON) - fitur terkunci: EXP-1 (normalisasi + TF-IDF unigram)

=== [M1-NB] M1-NB (Tahap B) ===
Accuracy: 0.7932 | Macro-F1: 0.6497
              precision    recall  f1-score   support

    positive     0.8018    0.9587    0.8733      1283
     neutral     1.0000    0.2217    0.3630       230
    negative     0.7545    0.6754    0.7127       687

    accuracy                         0.7932      2200
   macro avg     0.8521    0.6186    0.6497      2200
weighted avg     0.8078    0.7932    0.7698      2200

Confusion Matrix (baris=gold, kolom=prediksi; urutan positive, neutral, negative):
[[1230    0   53]
 [  81   51   98]
 [ 223    0  464]]

=== [M2-LR] M2-LR (Tahap B) ===
Accuracy: 0.8755 | Macro-F1: 0.8439
              precision    recall  f1-score   support

    positive     0.9032    0.9306    0.9167      1283
     neutral     0.9298    0.6913    0.7930       230
    negative     0.8105    0.8341    0.8221       687

    accuracy                         0.8755

### Kesamaan error lintas model (development set)

In [7]:
y_dev = df_dev["label"].values
wrong = {k: (v != y_dev) for k, v in dev_pred.items()}
n_wrong = sum(wrong.values())   # jumlah model yang salah per sampel
print("\nKESAMAAN ERROR (development set)")
for k in wrong: print(f"  {k}: {wrong[k].sum()} salah")
print("  salah di 3 model :", (n_wrong == 3).sum())
print("  salah di 2 model :", (n_wrong == 2).sum())
print("  salah di 1 model :", (n_wrong == 1).sum())
print("  Overlap NB&LR:", (wrong["M1-NB"] & wrong["M2-LR"]).sum(),
      "| LR&SVM:", (wrong["M2-LR"] & wrong["M3-SVM"]).sum(),
      "| NB&SVM:", (wrong["M1-NB"] & wrong["M3-SVM"]).sum())
print("\n5+ contoh error lintas model:")
idx = np.where(n_wrong >= 1)[0][:8]
for i in idx:
    print(f"- {df_dev['text'].iloc[i][:110]}")
    print(f"  gold={y_dev[i]} | NB={dev_pred['M1-NB'][i]} | LR={dev_pred['M2-LR'][i]} | SVM={dev_pred['M3-SVM'][i]}")


KESAMAAN ERROR (development set)
  M1-NB: 455 salah
  M2-LR: 274 salah
  M3-SVM: 275 salah
  salah di 3 model : 188
  salah di 2 model : 91
  salah di 1 model : 258
  Overlap NB&LR: 220 | LR&SVM: 229 | NB&SVM: 206

5+ contoh error lintas model:
- saya jadi semakin tidak suka jokowi
  gold=negative | NB=positive | LR=negative | SVM=negative
- cucu bung hatta tidak suka lo dengan pernyataan anda menyamakan prabowo dengan eyang karno dan sandiaga dengan
  gold=negative | NB=negative | LR=positive | SVM=negative
- gue berharap semoga yang lagi menyinyirkan korea ini tidak pakai handphone samsung ya
  gold=neutral | NB=negative | LR=negative | SVM=negative
- tempat ini mengimingi makan steak dan daging dengan harga murah . namun kenyataan nya tidak murah jika dibandi
  gold=negative | NB=positive | LR=positive | SVM=positive
- prabowo sama yenny saja ditolak . sabaar ya roh . memang tidak ada waktu yang tepat untuk sebuah penolakan .
  gold=negative | NB=positive | LR=negative | SVM=negati

## Evaluasi final dan daftar error (final evaluation set)
Pilih minimal 10 error dari daftar, lalu beri kategori secara manual di laporan.

In [8]:
best_name = max(["M1-NB","M2-LR","M3-SVM"],
                key=lambda k: precision_recall_fscore_support(y_dev, dev_pred[k], average="macro", zero_division=0)[2])
best_model = models[best_name]
print("\nModel final (Macro-F1 dev tertinggi):", best_name)
X_eval_norm = df_final_eval["text"].apply(prep_norm)
eval_pred = best_model.predict(vec_1.transform(X_eval_norm))
evaluate_pipeline(df_final_eval["label"], eval_pred, "FINAL-"+best_name, best_name+" pada final eval", stage="Final Evaluation")
df_eval_res = df_final_eval.copy(); df_eval_res["pred"] = eval_pred
errors = df_eval_res[df_eval_res["label"] != df_eval_res["pred"]]
print(f"Total salah: {len(errors)} / {len(df_eval_res)} (error rate {len(errors)/len(df_eval_res):.4f})")
print(errors.head(12).to_string())


Model final (Macro-F1 dev tertinggi): M3-SVM

=== [FINAL-M3-SVM] M3-SVM pada final eval (Final Evaluation) ===
Accuracy: 0.8690 | Macro-F1: 0.8322
              precision    recall  f1-score   support

    positive     0.9001    0.9197    0.9098       735
     neutral     0.8426    0.6947    0.7615       131
    negative     0.8180    0.8325    0.8252       394

    accuracy                         0.8690      1260
   macro avg     0.8536    0.8156    0.8322      1260
weighted avg     0.8685    0.8690    0.8679      1260

Confusion Matrix (baris=gold, kolom=prediksi; urutan positive, neutral, negative):
[[676  10  49]
 [ 16  91  24]
 [ 59   7 328]]
Total salah: 165 / 1260 (error rate 0.1310)
                                                                                                                                                                                                                                                                                                          

## Bagian 6. Trace satu prediksi

In [9]:
s_raw  = df_dev.iloc[0]["text"]; s_prep = X_dev_1.iloc[0]
s_vec  = vec_1.transform([s_prep]); fn = np.array(vec_1.get_feature_names_out())
lr = models["M2-LR"]; nz = s_vec.nonzero()[1]
print("\nTRACE\nRaw  :", s_raw, "\nPrep :", s_prep)
for k, cls in enumerate(lr.classes_):
    z = lr.intercept_[k] + sum(s_vec[0, j] * lr.coef_[k, j] for j in nz)
    print(f"  z[{cls}] = b({lr.intercept_[k]:+.4f}) + sum(w*x) = {z:+.4f}")
pred_lr = lr.predict(s_vec)[0]; k = list(lr.classes_).index(pred_lr)
tr = pd.DataFrame([(fn[j], s_vec[0,j], lr.coef_[k,j], s_vec[0,j]*lr.coef_[k,j]) for j in nz],
                  columns=["fitur x_i","TF-IDF x_i","bobot w_i","x_i*w_i"]).sort_values("x_i*w_i", ascending=False)
print("Prediksi LR:", pred_lr); print(tr.head(5).to_string(index=False))
# model final (jika bukan LR): decision score per kelas pada contoh yang sama
if best_name != "M2-LR" and hasattr(best_model, "coef_"):
    for kk, cls in enumerate(best_model.classes_):
        z = best_model.intercept_[kk] + sum(s_vec[0, j] * best_model.coef_[kk, j] for j in nz)
        print(f"  decision[{cls}] = {z:+.4f}")
    print("Prediksi model final:", best_model.predict(s_vec)[0])


TRACE
Raw  : bagaimana jokoer . sparatis dihai kok diam ? biasanya kalau ada yang demo kalian koar koar seperti kepanasan . payah ! cupu ! perjuangan 
Prep : bagaimana jokoer . sparatis dihai kok diam ? biasanya kalau ada yang demo kalian koar koar seperti kepanasan . payah ! cupu ! perjuangan
  z[negative] = b(+0.0899) + sum(w*x) = +1.8259
  z[neutral] = b(+0.2420) + sum(w*x) = -0.9877
  z[positive] = b(-0.3319) + sum(w*x) = -0.8382
Prediksi LR: negative
fitur x_i  TF-IDF x_i  bobot w_i  x_i*w_i
    payah    0.245404   1.674623 0.410959
bagaimana    0.217058   1.326498 0.287927
      kok    0.185760   1.425772 0.264852
   kalian    0.210116   1.201208 0.252393
     koar    0.619634   0.387399 0.240045
  decision[negative] = +1.0978
  decision[neutral] = -1.3092
  decision[positive] = -1.1950
Prediksi model final: negative


## Audit bahasa dan Bagian 8. Analisis bias

In [10]:
NEG = {"tidak","bukan","belum","tanpa","jangan","tak","gak","ga","nggak","ngga","gk","tdk"}
SLANG = set(slang_dict) | {"bgst","ngga","nggak","aja","udah","banget"}
import re as _re
def stats(df, name):
    w = df["text"].str.split()
    print(f"\n[{name}] n={len(df)}")
    print("  mengandung negasi :", w.apply(lambda t: bool(NEG & set(t))).sum())
    print("  mengandung slang  :", w.apply(lambda t: bool(SLANG & set(t))).sum())
    print("  token emoji/emotikon (__x__ / *face):", df["text"].str.contains(r"__\w+__|\w*face\b").sum())
    print("  mengandung tanda tanya / seru:", df["text"].str.contains(r"\?|!").sum())
stats(df_train, "Train"); stats(df_final_eval, "Final eval")
for nm, d in [("Raw train", df_raw_train), ("Dev", df_dev), ("Final eval", df_final_eval)]:
    print(nm, d["label"].value_counts().to_dict())

e = df_eval_res.copy(); e["n_words"] = e["text"].str.split().apply(len)
e["has_neg"]   = e["text"].str.split().apply(lambda t: bool(NEG & set(t)))
e["has_slang"] = e["text"].str.split().apply(lambda t: bool(SLANG & set(t)))
subsets = {"pendek (<=10 kata)": e["n_words"] <= 10, "panjang (>10 kata)": e["n_words"] > 10,
           "dengan negasi": e["has_neg"], "tanpa negasi": ~e["has_neg"],
           "dengan slang": e["has_slang"], "tanpa slang": ~e["has_slang"]}
print("\nANALISIS BIAS (final eval):")
rows = []
for nm, mask in subsets.items():
    d = e[mask]
    mf = precision_recall_fscore_support(d["label"], d["pred"], average="macro", labels=LABELS, zero_division=0)[2]
    rows.append((nm, len(d), round((d["label"]!=d["pred"]).mean(),4), round(mf,4)))
print(pd.DataFrame(rows, columns=["subset","n","error_rate","macro_F1"]).to_string(index=False))


[Train] n=8800
  mengandung negasi : 3339
  mengandung slang  : 1006
  token emoji/emotikon (__x__ / *face): 38
  mengandung tanda tanya / seru: 881

[Final eval] n=1260
  mengandung negasi : 480
  mengandung slang  : 141
  token emoji/emotikon (__x__ / *face): 4
  mengandung tanda tanya / seru: 137
Raw train {'positive': 6416, 'negative': 3436, 'neutral': 1148}
Dev {'positive': 1283, 'negative': 687, 'neutral': 230}
Final eval {'positive': 735, 'negative': 394, 'neutral': 131}

ANALISIS BIAS (final eval):
            subset    n  error_rate  macro_F1
pendek (<=10 kata)  187      0.2674    0.7262
panjang (>10 kata) 1073      0.1072    0.8389
     dengan negasi  480      0.1521    0.7943
      tanpa negasi  780      0.1179    0.8460
      dengan slang  141      0.0780    0.5902
       tanpa slang 1119      0.1376    0.8318


## Simpan results.csv

In [11]:
pd.DataFrame(results_list).to_csv("results.csv", index=False)
print("\nresults.csv disimpan:", len(results_list), "baris")
pd.DataFrame(results_list)


results.csv disimpan: 13 baris


,Stage,Exp_ID,Description,Accuracy,Macro_F1,Prec_Pos,Rec_Pos,F1_Pos,Prec_Neu,Rec_Neu,F1_Neu,Prec_Neg,Rec_Neg,F1_Neg
0,Tahap A,EXP-0,Baseline Minimal + Unigram + LogReg,0.8745,0.8428,0.9030,0.9291,0.9159,0.9244,0.6913,0.7910,0.8093,0.8341,0.8215
1,Tahap A,EXP-1,EXP-0 + Normalization,0.8755,0.8439,0.9032,0.9306,0.9167,0.9298,0.6913,0.7930,0.8105,0.8341,0.8221
2,Tahap A,EXP-2,EXP-1 + Stopword,0.8645,0.8246,0.8935,0.9283,0.9106,0.9299,0.6348,0.7545,0.7958,0.8224,0.8089
3,Tahap A,EXP-3,EXP-1 + Stemming,0.8718,0.8383,0.8993,0.9260,0.9124,0.8798,0.7000,0.7797,0.8175,0.8282,0.8228
4,Tahap A,EXP-4,"Best Preprocess + TF-IDF (1,2) N-gram",0.8727,0.8158,0.8973,0.9462,0.9211,0.9692,0.5478,0.7000,0.8089,0.8443,0.8262
5,Tahap A,EXP-2b,EXP-1 + Stopword (negasi ikut dibuang),0.8532,0.8001,0.8890,0.9299,0.9090,0.9041,0.5739,0.7021,0.7753,0.8035,0.7891
6,Tahap B,M1-NB,M1-NB,0.7932,0.6497,0.8018,0.9587,0.8733,1.0000,0.2217,0.3630,0.7545,0.6754,0.7127
7,Tahap B,M2-LR,M2-LR,0.8755,0.8439,0.9032,0.9306,0.9167,0.9298,0.6913,0.7930,0.8105,0.8341,0.8221
8,Tahap B,M3-SVM,M3-SVM,0.8750,0.8486,0.9013,0.9252,0.9131,0.8947,0.7391,0.8095,0.8196,0.8268,0.8232
9,Tahap B (tambahan),M1-NB-bigram,M1-NB fitur EXP-4,0.7395,0.5332,0.7161,0.9852,0.8294,1.0000,0.0783,0.1452,0.8273,0.5022,0.6250
